# Homework #5

## Ensemble learning

This colaboratory contains Homework #5 of the Machine Learning course, which is due **April 4, midnight (23:59 EEST time)**. To complete the homework, extract **(File -> Download .ipynb)** and submit to the course webpage.


## Submission's rules:

1.   Please, submit only .ipynb that you extract from the Colaboratory.
2. Run your homework exercises before submitting (output should be present, preferably restart the kernel and press run all the cells).
3. Do not change the description of tasks in red (even if there is a typo|mistake|etc).
4. Please, make sure to avoid unnecessary long printouts.
5. Each task should be solved right under the question of the task and not elsewhere.
6. Solutions to both regular and bonus exercises should be submitted in one IPYNB file.

Please, steer clear of copying someone else's work. If you discuss assignments with anyone in the course, please, mention their names here:

Pooh

##List of Homework's exercises:

1.   [Ex1](#scrollTo=ux5PBYkbwewj) - 2 points
2.   [Ex2](#scrollTo=Z5C3GX9eXFF_) - 4 points
3.   [Ex3](#scrollTo=uHQscVCD7rM0) - 4 points
4.   [Bonus 1](#scrollTo=iJeG56t9-jWI) - 1 point
5.   [Bonus 2](#scrollTo=cLoHOqAcu0Q9) - up to 3 points
6.   [Bonus 3](#scrollTo=jdZkblZW7bEp) - up to 3 points (based on quality of presentation)




---


## Homework exercise 1: implement and train a bagging classifier with 3 K-NN models as estimators (2 points)


<font color='red'> In this exercise you will need to use `classify_knn` function from the first practice session to train three different KNN models on three resamples of this dataset. </font>


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def create_random_2c_data (D, N):
  """
  Function create_random_2c_data generates two sets of D dimensional
  points (N points each), one for each class. The first set is sampled from D
  dimensional Gaussian distribution with mean 0 and standard deviation 1. The
  second set is generated from the distribution, with mean 1 and standard
  deviation 1.
  """
  # Generating N points for the first class
  mu_vec1 = np.zeros(D) # creates a vector of zeros, these are averages across each dimension
  cov_mat1 = np.eye(D) # creates a diagonal matrix of size D x D, all values except diagonal are 0
  class1_sample = np.random.multivariate_normal(mu_vec1, cov_mat1, N)

  # The same stuff as above, just averages are shifted into 1
  mu_vec2 = np.ones(D) # creates a vector of ones
  cov_mat2 = np.eye(D)
  class2_sample = np.random.multivariate_normal(mu_vec2, cov_mat2, N)

  # a lot of boring things....
  # gluing together two matrices generated above
  data = pd.DataFrame(np.concatenate((class1_sample, class2_sample)))

  # Create names for columns
  data.columns = [ 'x' + str(i) for i in (np.arange(D)+1)]

  # Create a class column
  data['class'] = np.concatenate((np.repeat(0, N), np.repeat(1, N)))

  # This is important for plotting and modelling
  data['class'] = data['class'].astype('category')

  return data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
np.random.seed(2342347823) # random seed, this number was random, no need to build conspiracies around it

D = 2 # two dimensions
N = 100 # points per class

whole_data = create_random_2c_data(D, N)

# Randomly splitting data into train (60%) and validation (40%)
train, val = train_test_split(whole_data, random_state = 111, test_size = 0.40)

n_bootstraps = 3
np.random.seed(1111)

# creating resamples
resamples = [resample(train, n_samples = len(train), replace=True).index.values for i in range(n_bootstraps)]

# first resample
train_resample1 = train.loc[resamples[0]]

# second resample
train_resample2 = train.loc[resamples[1]]

# third resample
train_resample3 = train.loc[resamples[2]]

<font color='red'> Here, I just convert pandas DataFrame into Numpy arrays that are easier to use list comprehension mechanisms on. </font>

In [ ]:
train1 = np.asarray(train_resample1[['x1','x2']])
labels1 = np.asarray(train_resample1[['class']]).reshape((train_resample1.shape[0]))

train2 = np.asarray(train_resample2[['x1','x2']])
labels2 = np.asarray(train_resample2[['class']]).reshape((train_resample2.shape[0]))

train3 = np.asarray(train_resample3[['x1','x2']])
labels3 = np.asarray(train_resample3[['class']]).reshape((train_resample3.shape[0]))

val_points = np.asarray(val[['x1','x2']])
val_labels = np.asarray(val[['class']]).reshape((val.shape[0]))

<font color='red'>  **(Homework exercise 1- a)** Copy and adapt `classify_knn` function from the first homework and practice session to operate on 2D points. **(0.5 points)**</font>

In [ ]:
def dist(point1, point2): # function dist is also from the first practice session
  # sum of squared coordinate-wise differences under sqrt
  return(np.sqrt(np.sum((point2 - point1)**2)))

def classify_knn(val_point, k, train, labels):

  ##### YOUR CODE STARTS #####
  all_distances = []
  nearest_neighbours = []

  loc_dist = 0

  for i in train:
    loc_dist = dist(val_point,i)
    all_distances.append(loc_dist)

  knn_idx = []

  for i in range(k):
    idx = np.argmin(all_distances)

    knn_idx.append(idx)

    all_distances[idx] = np.inf

  for i in knn_idx:
    nearest_neighbours.append(labels[i])

  counts = np.bincount(nearest_neighbours)

  prediction = np.argmax(counts)

  predicted_classes = []

  for i in range(10):
    idx = np.argmax(counts)
    predicted_classes.append(idx)
    counts[idx] = -1

  ##### YOUR CODE ENDS #####

  return prediction

<font color='red'> Test that the function was adapted correctly by running the following example </font>

In [ ]:
val_point = val_points[1]
print(f'predicted class of the first point is {classify_knn(val_point, 5,  train1, labels1)}, while the true class is {val_labels[1]}')

predicted class of the first point is 0, while the true class is 0


<font color='red'> **(Homework exercise 1- b)** Classify each point from the validation set using `classify_knn` function. Use different resamples and list comprehension. Fix `k` at 5. **(1 point)**</font>


In [ ]:
k = 5

# Use three K-NN models that work on three different resamples

##### YOUR CODE STARTS #####
val['knn1'] = [classify_knn(val_point, 5, train1, labels1) for val_point in val_points]
val['knn2'] = [classify_knn(val_point, 5, train2, labels2) for val_point in val_points]
val['knn3'] = [classify_knn(val_point, 5, train3, labels3) for val_point in val_points]
##### YOUR CODE ENDS #####

<font color='red'> Below aggregate individual predictions using the majority vote approach</font>

In [ ]:
##### YOUR CODE STARTS #####
val['knn_bagging'] = val[['knn1', 'knn2', 'knn3']].mode(axis = 1)
##### YOUR CODE ENDS #####

print(f"Accuracy of hand made bagged ensemble with 3 KNNs is {np.sum(val['knn_bagging'] == val['class'])/len(val[['class']])*100}%")

Accuracy of hand made bagged ensemble with 3 KNNs is 66.25%


<font color='red'> **(Homework exercise 1- c)** Use sklearn `BaggingClassifier` to implement analogous model that uses KNeighborsClassifier as an estimator (with k = 5). Don't forget to use a random state for reproducibility.

Assess its performance on the same validation set and display it. **(0.5 points)**</font>


In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier

##### YOUR CODE STARTS #####
knn_begger = BaggingClassifier(estimator = KNeighborsClassifier(n_neighbors = 5), n_estimators = 3, random_state=112)
knn_begger.fit(train[['x1', 'x2']], train['class'])
##### YOUR CODE ENDS #####
print(f"Accuracy of sklearn bagging with {3} KNNs {knn_begger.score(val[['x1', 'x2']], val[['class']])*100}%")

KeyError: "None of [Index(['x1', 'x2'], dtype='object')] are in the [columns]"

## Homework exercise 2: eXtreme Gradient Boosting (XGBoost) (4 points)

<font color='red'> Let's finally build for ourselves a new shiny XGBoost model, the most popular algorithm for Kaggle competitions. </font>
<font color='red'>  Since XGBoost truly shines on tabular data, we are going to use the Thyroid Disease Dataset for training an XGboost model. Further details about this dataset can be found at
 [here](https://www.openml.org/search?type=data&status=active&id=40475). </font>


In [ ]:
from sklearn.datasets import fetch_openml

data = fetch_openml(name="thyroid-allhyper")

In [ ]:
#Let's take a look at data
data.data

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26
0,41.0,1,1,1,1,1,1,1,1,1,...,2,1.30000,2,2.500000,2,125.0,2,1.140000,2,109.000000
1,23.0,1,1,1,1,1,1,1,1,1,...,2,4.10000,2,2.000000,2,102.0,1,0.997912,1,110.787984
2,46.0,2,1,1,1,1,1,1,1,1,...,2,0.98000,1,2.024966,2,109.0,2,0.910000,2,120.000000
3,70.0,1,2,1,1,1,1,1,1,1,...,2,0.16000,2,1.900000,2,175.0,1,0.997912,1,110.787984
4,70.0,1,1,1,1,1,1,1,1,1,...,2,0.72000,2,1.200000,2,61.0,2,0.870000,2,70.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2795,70.0,2,1,1,1,1,1,1,1,1,...,2,2.70000,1,2.024966,2,155.0,2,1.050000,2,148.000000
2796,73.0,2,1,2,1,1,1,1,1,1,...,1,4.67215,2,0.700000,2,63.0,2,0.880000,2,72.000000
2797,75.0,2,1,1,1,1,1,1,1,1,...,1,4.67215,1,2.024966,2,147.0,2,0.800000,2,183.000000
2798,60.0,1,1,1,1,1,1,1,1,1,...,2,1.40000,1,2.024966,2,100.0,2,0.830000,2,121.000000


In [ ]:
# Types of classes
data.target

,Class
0,3
1,1
2,1
3,1
4,5
...,...
2795,5
2796,1
2797,1
2798,1


In [ ]:
# Number of instances for each class
data.target.value_counts()

,count
Class,
1,1632
5,771
3,275
2,91
4,31


<font color='red'> **(Homework exercise 2- a)** Complete the preprocessing steps and train the XGBoost model for the Thyroid Disease classification task.
Find additional information regarding the training of an XGBoost model [here](https://xgboost.readthedocs.io/en/latest/python/python_intro.html) **(1.5 points)** </font>

In [ ]:
# Initial data preparation, which involves spliting the dataset into training, validation, and test sets
# Use 80/10/10 datasplit

##### YOUR CODE STARTS #####
n = data.data.shape[0]

train_end = int(n * 0.8)
val_end = int(n * 0.9)

train = data.data[:train_end]
train_labels = data.target[:train_end].astype(int) - 1

val = data.data[train_end:val_end]
val_labels = data.target[train_end:val_end].astype(int) - 1

test = data.data[val_end:]
test_labels = data.target[val_end:].astype(int) - 1

##### YOUR CODE ENDS #####

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score

##### YOUR CODE STARTS #####

# XGBoost wants data to be wrapped into special formats
dtrain = xgb.DMatrix(train, label = train_labels, enable_categorical=True)
dval = xgb.DMatrix(val, label = val_labels, enable_categorical=True)
dtest = xgb.DMatrix(test, label = test_labels, enable_categorical=True)

# Define the XGBoost parameters (you can add more parameters if you want)
params = {
    "objective": 'multi:softmax',
    "num_class": 5,
    "eval_metric": 'mlogloss'
}

# Train for training and val for validation
eval_list = [(dtrain, "train"), (dval, "validation")]

# Train the XGBoost model
model = xgb.train(params = params, dtrain = dtrain, evals = eval_list, num_boost_round = 100, early_stopping_rounds = 3)

##### YOUR CODE ENDS #####

[0]	train-mlogloss:0.89034	validation-mlogloss:0.91605
[1]	train-mlogloss:0.80687	validation-mlogloss:0.86184
[2]	train-mlogloss:0.74313	validation-mlogloss:0.81792
[3]	train-mlogloss:0.69586	validation-mlogloss:0.79344
[4]	train-mlogloss:0.65708	validation-mlogloss:0.77418
[5]	train-mlogloss:0.62831	validation-mlogloss:0.76539
[6]	train-mlogloss:0.60163	validation-mlogloss:0.75269
[7]	train-mlogloss:0.57871	validation-mlogloss:0.74611
[8]	train-mlogloss:0.55989	validation-mlogloss:0.73711
[9]	train-mlogloss:0.54238	validation-mlogloss:0.72533
[10]	train-mlogloss:0.52802	validation-mlogloss:0.72346
[11]	train-mlogloss:0.51497	validation-mlogloss:0.72376
[12]	train-mlogloss:0.50425	validation-mlogloss:0.72254
[13]	train-mlogloss:0.49320	validation-mlogloss:0.71851
[14]	train-mlogloss:0.48258	validation-mlogloss:0.72120
[15]	train-mlogloss:0.47419	validation-mlogloss:0.71694
[16]	train-mlogloss:0.46520	validation-mlogloss:0.71736
[17]	train-mlogloss:0.45870	validation-mlogloss:0.71494
[1

<font color='red'> **(Homework exercise 2- b)**  Make predictions on the test data and evaluate the model **(0.5 points)** </font>

In [ ]:
##### YOUR CODE STARTS #####
res = model.predict(dtest)

print(np.sum(res == test_labels) / len(test_labels))

##### YOUR CODE ENDS #####

0.7607142857142857


<font color='red'> **(Homework exercise 2- c)**  In machine learning, feature importance scores are used to determine the relative importance of each feature in a dataset when building a predictive model. Such scores can be derived from a variety of different models, including ensembles. Explore feature importance of the resulting XGBoost model and report the top five most important features **(0.5 points)** </font>



In [ ]:
##### YOUR CODE STARTS #####
importance = model.get_score(importance_type = "weight")

print(sorted(importance.values(), reverse=True))
print(importance)
# ax = importance.values()
# print(sum(ax))
##### YOUR CODE ENDS #####

[425.0, 367.0, 307.0, 296.0, 275.0, 229.0, 67.0, 46.0, 33.0, 28.0, 27.0, 25.0, 22.0, 20.0, 19.0, 14.0, 13.0, 13.0, 10.0, 10.0, 9.0, 8.0, 5.0, 3.0]
{'V1': 425.0, 'V2': 67.0, 'V3': 46.0, 'V4': 3.0, 'V5': 10.0, 'V6': 28.0, 'V7': 13.0, 'V9': 20.0, 'V10': 14.0, 'V11': 19.0, 'V12': 13.0, 'V13': 5.0, 'V14': 22.0, 'V16': 27.0, 'V17': 25.0, 'V18': 367.0, 'V19': 33.0, 'V20': 229.0, 'V21': 10.0, 'V22': 307.0, 'V23': 8.0, 'V24': 275.0, 'V25': 9.0, 'V26': 296.0}


<font color='red'> **(Homework exercise 2- d)** Train Adaptive Boosting, Gradient Boosting and a simple KNN model from sklearn (KNeighborsClassifier) on the same training data and evaluate on the same test data. For each model use the default hyperparameters (e.g. `n_estimators` or `n_neighbors`). If you do not want to use default parameters, you can use `cross_val_score` to pick the best values for the hyperparameters using training data. Compare the performance of these three models and XGBoost and draw conclusions in a separate text cell  **(1.5 points)** </font>

In [ ]:
# AdaBoostClassifier
%%time
##### YOUR CODE STARTS #####
from sklearn.ensemble import AdaBoostClassifier

model = AdaBoostClassifier()

model.fit(train, train_labels)

res = model.predict(test)

print(f'acc:{np.sum(res == test_labels)/ len(test_labels)}')

##### YOUR CODE ENDS #####

acc:0.6892857142857143
CPU times: user 1.01 s, sys: 4.01 ms, total: 1.02 s
Wall time: 1.16 s


In [ ]:
# GradientBoostingClassifier
%%time
# might take considerable time if trained with default number of n_estimators
##### YOUR CODE STARTS #####
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier()

model.fit(train, train_labels)

res = model.predict(test)

print(f'acc:{np.sum(res == test_labels)/ len(test_labels)}')

##### YOUR CODE ENDS #####

acc:0.7642857142857142
CPU times: user 3.63 s, sys: 3.45 ms, total: 3.64 s
Wall time: 6.43 s


In [ ]:
# KNeighborsClassifier
%%time
##### YOUR CODE STARTS #####
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier()

model.fit(train, train_labels)

res = model.predict(test)

print(f'acc:{np.sum(res == test_labels)/ len(test_labels)}')


##### YOUR CODE ENDS #####

acc:0.5714285714285714
CPU times: user 53.2 ms, sys: 4.89 ms, total: 58.1 ms
Wall time: 41.2 ms


<font color='red'> How these models compare with each other and to XGBoost? Can you try to elaborate on this difference? </font>

In [ ]:
# Write your comment here:
# The results show that the highest accuracy was achieved by the GradientBoostingClassifier (0.7607), which is equal to the performance of XGBoost.
# The lowest accuracy was obtained by the KNeighborsClassifier (0.5714), as it is the simplest model among those considered.
# The other models are ensemble methods, which explains their better performance compared to KNN. The AdaBoostClassifier demonstrates a moderate result (0.6893)

## Homework exercise 3: Kaggle in-class to predict whether a patient will be readmitted to the hospital within 30 days after discharge (4 points)
<font color='red'> In this exercise we will see how well ensembles work in the real-life problems. Here we are going to work with medical data, and more specifically we will attempt to predict whether a patient will be readmitted to the hospital within 30 days after discharge based on numerical and categorical features. To this end, we have created a corresponding Kaggle in-class competition: https://www.kaggle.com/t/cfc5280da3b847ab8314140f3b5fd61a (this is invitation link). </font>

<font color='red'> **(Homework exercise 3- a)** Getting started: follow the code provided with the Kaggle competition (`sample_notebook.ipynb`) to make the first sample submission using `random_predict` function, Kaggle API or by manually uploading submission file to https://www.kaggle.com/competitions/ml-kse-2026. Report your score. **(0.5 points)** </font>

In [ ]:
import pandas as pd
import numpy as np
import random

<font color='red'> Load `train.csv` and `test.csv` from https://www.kaggle.com/competitions/ml-kse-2026

In [ ]:
from google.colab import files
files.upload();

Saving train.csv to train (1).csv


In [ ]:
files.upload();

Saving test.csv to test (1).csv


In [ ]:
##### YOUR CODE STARTS #####
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
##### YOUR CODE ENDS #####

<font color='red'> Insert your random guesser below:

In [ ]:
##### YOUR CODE STARTS #####
predict = [np.random.randint(0, 2) for i in range(test.shape[0])]
##### YOUR CODE ENDS #####

<font color='red'> Create a sample submission below

In [ ]:
##### YOUR CODE STARTS #####
submission = test[['id']].copy()

submission['target'] = predict

output_path = 'submission_from_notebook.csv'

submission.to_csv(output_path, index=False)
##### YOUR CODE ENDS #####

NameError: name 'predict' is not defined

In [ ]:
!kaggle competitions submit -c ml-kse-2026 -f submission_from_notebook.csv -m "Random guesser submission"

100% 222k/222k [00:00<00:00, 630kB/s]
Successfully submitted to ML@KSE2026

<font color='red'> You can either use Kaggle API below to submit submission file or download file and submit it manually to Kaggle via web interface. </font>

<font color='red'> You need to have an account on Kaggle.com, before you proceed. When you access your kaggle profile, you need to download your API Token from kaggle. It's very easy:
1. Click on your profile icon
2. Go to **Account**
3. In **API** you press **Create Legacy API token**

**Optionally:** You can create not a Legacy type, altough the token preparation in the notebook process will differ

<font color='red'> Now we load the file **kaggle.json** that you have downloaded, into this notebook: </font>

In [ ]:
files.upload(); # upload your kaggle.json file

Saving kaggle.json to kaggle.json


<font color='red'> The next cell moves the file into a separate folder, sets secure access for it and configures your Kaggle profile for this notebook.</font>

In [ ]:
import json

!mkdir /root/.kaggle/
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!kaggle config set -n path -v{/content}

- path is now set to: {/content}


<font color='red'> Assuming that you have accepted the rules of the competition, you should be able to make a submission using the following code: </font>

In [ ]:
!kaggle competitions submit -c ml-kse-2026 -f submission_from_notebook.csv -m "2"

100% 222k/222k [00:00<00:00, 1.14MB/s]
400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission


<font color='red'> What was the score you received? </font>

In [ ]:
# Report your result here:
#  0.40947

<font color='red'> **(Homework exercise 3- b)** Take off: make two more submissions with at least two different ensemble models we have studied. Make sure to perform all necessary data pre-processing steps to accommodate the requirements of these ensemble models (e.g. data imputation and restructuring). To get full points, shortly summarise your pre-processing, models you have selected and rational behind this choice, and finally report your scores. **(2 points)** </font>

In [ ]:
##### YOUR CODE STARTS #####
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_x = train.iloc[:, 1:-1]
train_y = train['target']

train_x = scaler.fit_transform(train_x)

test_x = test.iloc[:, 1:]
test_x = scaler.fit_transform(test_x)


estimators = [
    ('decision_tree', DecisionTreeClassifier(random_state = 42)),
    ('log_reg', LogisticRegression(max_iter = 10000, solver = 'lbfgs'))
]

model = StackingClassifier(estimators)

model.fit(train_x, train_y)

res = model.predict(test_x)

print(sum(res == 1))
##### YOUR CODE ENDS #####

47


In [ ]:
submission = test[['id']].copy()

submission['target'] = res

output_path = 'submission_from_notebook.csv'

submission.to_csv(output_path, index=False)

In [ ]:
print(train.describe())

                f_1           f_2           f_3           f_4           f_5  \
count  65165.000000  65165.000000  65165.000000  65165.000000  65165.000000   
mean       2.022374      3.710872      5.752413      4.403376     43.124806   
std        1.447320      5.274660      4.050275      2.989682     19.737365   
min        1.000000      1.000000      1.000000      1.000000      1.000000   
25%        1.000000      1.000000      1.000000      2.000000     31.000000   
50%        1.000000      1.000000      7.000000      4.000000     44.000000   
75%        3.000000      3.000000      7.000000      6.000000     57.000000   
max        8.000000     28.000000     25.000000     14.000000    132.000000   

                f_6           f_7           f_8           f_9          f_10  \
count  65165.000000  65165.000000  65165.000000  65165.000000  65165.000000   
mean       1.339461     16.024261      0.371963      0.200890      0.635188   
std        1.706360      8.103660      1.264854    

In [ ]:
print(test.columns)

Index(['id', 'f_1', 'f_2', 'f_3', 'f_4', 'f_5', 'f_6', 'f_7', 'f_8', 'f_9',
       'f_10', 'f_11', 'fc_1', 'fc_2', 'fc_3', 'fc_4', 'fc_5', 'fc_6', 'fc_7',
       'fc_8', 'fc_9', 'fc_10', 'fc_11', 'fc_12', 'fc_13', 'fc_14', 'fc_15',
       'fc_16', 'fc_17', 'fc_18', 'fc_19', 'fc_20', 'fc_21', 'fc_22', 'fc_23',
       'fc_24', 'fc_25', 'fc_26', 'fc_27', 'fc_28', 'fc_29', 'fc_30', 'fc_31',
       'fc_32', 'fc_33', 'fc_34', 'fc_35', 'fc_36'],
      dtype='object')


<font color='red'> Summarise your pre-processing steps, models selected (why these?): </font>

In [ ]:
# Add your comment here:


<font color='red'> **(Homework exercise 3- c)** Reaching superiority: tune the above models or train a new one (does not have to be ensemble) such that you beat Mr.Smart (`mr_smart.csv`) benchmark. If one or both of your models from 3-b are already superior to Mr.Smart, then you should beat them instead. In order to claim the point in this exercise, describe the model, data pre-processing steps and report your score (which should be higher than Mr Smart's and your previous' models). **(1.5 points)** </font>

In [ ]:
##### YOUR CODE STARTS #####
over = RandomOverSampler(random_state = 42, sampling_strategy=0.6)

train_x_ov, train_y_ov = over.fit_resample(train_x, train_y)

under = RandomUnderSampler(sampling_strategy = 0.6, random_state = 42)
train_x_over, train_y_over = under.fit_resample(train_x_ov, train_y_ov)

model = xgb.XGBClassifier(n_estimators = 300, max_depth = 3, tree_method = 'hist', random_state = 42, learning_rate = 0.1)

model.fit(train_x_over, train_y_over)

res = model.predict(test_x)
##### YOUR CODE ENDS #####

<font color='red'> Summarise your pre-processing steps, models selected (why these?), and the "special sauce" (what you changed compared to 3-b) below: </font>

In [ ]:
# Add your comment here: I did all perfomance with data which I knew: data augmentation
# (Tomek links, smote random ...), feature engineering ......The best performance was
# achieved using a data balancing strategy that combines random oversampling with
# random undersampling.

## Bonus exercise 3 (up to 3 bonus points depending on presentation): Discrete AdaBoost from scratch



<font color='red'> In the class we have discussed a simplified version of the AdaBoost, the goal of this task is to implement an unabrdiged version of the Discrete AdaBoost classifier following the instructions from the paper [Friedman et al (2000)](https://hastie.su.domains/Papers/AdditiveLogisticRegression/alr.pdf). In a nutshell, AdaBoost classifier assigns initial weights to training examples, fits a weak classifier, and updates the data and classifier weights based on classification accuracy. The process is repeated for each weak estimator. The final strong classifier combines weak classifiers with different weights.</font>

<font color='red'> Implement AdaBoost classifier.

Tips:

- use `DecisionTreeClassifier` from `sklearn` as a weak classifier
- avoid division by zero
- pay attention to the format of data labels.</font>



In [ ]:
from sklearn.tree import DecisionTreeClassifier


class AdaBoost:
  def __init__(self, n_estimators = 50):
    self.n_estimators = n_estimators
    self.estimator_weights = []
    self.models = []

  def fit(self, X, y):
    ##### YOUR CODE STARTS #####

    weight = np.array([1 / len(X) for i in range(len(X))])

    y = [-1 if label == 0 else 1 for label in y]

    for i in range(self.n_estimators):
      model = DecisionTreeClassifier(max_depth = 1)

      model.fit(X, y, sample_weight = weight)

      res = model.predict(X)

      nepravilni = res != y

      error_estimator = np.sum(nepravilni * weight)


      self.estimator_weights.append(np.log((1 - error_estimator) / error_estimator))

      self.models.append(model)


      cm = np.log((1-error_estimator) / error_estimator)

      # weight = [waga * np.exp(cm) if nepravilni[idx] else waga for idx, waga in enumerate(weight)]
      weight = weight * np.exp(cm * nepravilni)

      suma_wag = np.sum(weight)

      weight = weight / suma_wag

    ##### YOUR CODE ENDS #####

  def predict(self, X):
    ##### YOUR CODE STARTS #####

    res = []
    for i in range(self.n_estimators):
      model = self.models[i]

      res.append(model.predict(X))

    res = np.array([pred * w for pred, w in zip(res, self.estimator_weights)])

    res = np.sum(res, axis = 0)

    final_result = [1 if i > 0 else 0 for i in res]

    return np.array(final_result)


    ##### YOUR CODE ENDS #####


<font color='red'> Test your model on the dataset from exercise 3. Report accuracy on the test set and observe how the performance changes with the different number of estimators.</font>

In [ ]:
##### YOUR CODE STARTS #####
model = AdaBoost(n_estimators=50)

model.fit(train_x, train_y)

res = model.predict(test_x)

submission = test[['id']].copy()

submission['target'] = res.astype(int)

output_path = 'submission_from_notebook.csv'

submission.to_csv(output_path, index=False)

!kaggle competitions submit -c ml-kse-2026 -f submission_from_notebook.csv -m "ada from ourself"
##### YOUR CODE ENDS #####

100% 222k/222k [00:00<00:00, 938kB/s]
Successfully submitted to ML@KSE2026

<font color='red'> Use sklearn implementation of AdaBoost classifier with the same parameters as you used for your implementation. Report accuracy on the test set as well. </font>

In [ ]:
##### YOUR CODE STARTS #####
from sklearn.ensemble import AdaBoostClassifier

model = AdaBoostClassifier(n_estimators=50)

model.fit(train_x, train_y)

res = model.predict(test_x)

submission = test[['id']].copy()

submission['target'] = res.astype(int)

output_path = 'submission_from_notebook.csv'

submission.to_csv(output_path, index=False)

!kaggle competitions submit -c ml-kse-2026 -f submission_from_notebook.csv -m "ada from scikit-learn"

##### YOUR CODE ENDS #####

100% 222k/222k [00:00<00:00, 1.08MB/s]
Successfully submitted to ML@KSE2026

<font color='red'> Report difference in results, if there are any. Explain why sklearn's model behaves differently, you might want to take a look at sklearn's [code](https://github.com/scikit-learn/scikit-learn/blob/093e0cf14/sklearn/ensemble/_weight_boosting.py#L341) :)</font>

<font color='red'> Your answer:</font>

Your comments:

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files
files.upload();

Saving train.csv to train.csv


In [ ]:
files.upload();

Saving test.csv to test.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train['f_new'] = train['f_2'] * train['f_10']
test['f_new'] = test['f_2'] * test['f_10']


num_cols = [col for col in train.columns if col.startswith('f_')]
cat_cols = [col for col in train.columns if col.startswith('fc_')]

train_y = train['target']
train_x = train.drop(columns=['target', 'id'])

columns_on_train = train_x.columns
test_x = test.drop(columns=['id'])

scaler = StandardScaler()
train_num = scaler.fit_transform(train_x[num_cols])
test_num = scaler.transform(test_x[num_cols])

train_x = np.hstack([train_num, train_x[cat_cols].values])
test_x  = np.hstack([test_num, test_x[cat_cols].values])



In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_x = train.drop(columns=['target', 'id'])
train_y = train['target']

train_x = scaler.fit_transform(train_x)

test_x = test.drop(columns=['id'])
test_x = scaler.transform(test_x)

In [ ]:
from imblearn.over_sampling import SMOTE

over = SMOTE(random_state=42)

train_x_over, train_y_over = over.fit_resample(train_x, train_y)

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

estimators = [
    ('decision_tree', DecisionTreeClassifier(random_state = 42, max_depth = 1)),
    ('log_reg', LogisticRegression(max_iter = 10000, solver = 'lbfgs'))
]

model = StackingClassifier(estimators)

model.fit(train_x, train_y)

res = model.predict(X_test)


print(f1_score(res, y_test, average='macro'))

0.48008550051029186


In [ ]:
from sklearn.model_selection import cross_val_score

depth = [1, 3, 5, 7, 10, 15, 30, 50, 100]

for i in depth:
  model = DecisionTreeClassifier(random_state = 42, max_depth = i)

  scores = cross_val_score(model, train_x, train_y, cv=5, scoring='f1')

  print(f"depth = {i} score: {np.mean(scores)}")



depth = 1 score: 0.8873321568326556
depth = 3 score: 0.8870559349343973
depth = 5 score: 0.8867950586971534
depth = 7 score: 0.8787692779866492
depth = 10 score: 0.8685644134121077
depth = 15 score: 0.8523287040589272
depth = 30 score: 0.7912836645438501
depth = 50 score: 0.7857592265786849
depth = 100 score: 0.7849612522059387


In [ ]:
from sklearn.model_selection import cross_val_score

max_iter = [5, 10, 20, 30, 50, 70, 100, 200, 300, 500, 1000]

for i in max_iter:
  model = LogisticRegression(max_iter = i, solver = 'lbfgs')

  scores = cross_val_score(model, train_x, train_y, cv=5, scoring='f1')

  print(f"depth = {i} score: {np.mean(scores)}")

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.25, random_state=0)


over = SMOTE(random_state = 42)

train_x_over, train_y_over = over.fit_resample(X_train, y_train)

n_estim = [2, 3, 5, 8, 12, 15, 20, 30 ,50]

max_depth = [1, 3, 5, 10, 20, 50, 100]


for est in n_estim:
  for depth in max_depth:
    model1 = xgb.XGBClassifier(n_estimators = est, max_depth = depth, tree_method = 'hist')
    model2 = xgb.XGBClassifier(n_estimators = est, max_depth = depth, tree_method = 'hist')

    model1.fit(train_x_over, train_y_over)
    model2.fit(X_train, y_train)

    res1 = model1.predict(X_test)
    res2 = model2.predict(X_test)

    print(f"WithOver: n_estim = {est}, depth = {depth} score: {f1_score(res1, y_test, average='macro')}")
    print(f"n_estim = {est}, depth = {depth} score: {f1_score(res2, y_test, average='macro')}")


# # Train for training and val for validation
# eval_list = [(dtrain, "train"), (dval, "validation")]

# # Train the XGBoost model
# model = xgb.train(params = params, dtrain = dtrain, evals = eval_list, num_boost_round = 100, early_stopping_rounds = 3)

WithOver: n_estim = 2, depth = 1 score: 0.518316043094688
n_estim = 2, depth = 1 score: 0.4695233133628549
WithOver: n_estim = 2, depth = 3 score: 0.5580638049242228
n_estim = 2, depth = 3 score: 0.4695233133628549
WithOver: n_estim = 2, depth = 5 score: 0.5640836459640101
n_estim = 2, depth = 5 score: 0.4695233133628549
WithOver: n_estim = 2, depth = 10 score: 0.5440449109553053
n_estim = 2, depth = 10 score: 0.47420794390339654
WithOver: n_estim = 2, depth = 20 score: 0.5370915562536652
n_estim = 2, depth = 20 score: 0.47632591032062965
WithOver: n_estim = 2, depth = 50 score: 0.537799993245297
n_estim = 2, depth = 50 score: 0.47632591032062965
WithOver: n_estim = 2, depth = 100 score: 0.537799993245297
n_estim = 2, depth = 100 score: 0.47632591032062965
WithOver: n_estim = 3, depth = 1 score: 0.518316043094688
n_estim = 3, depth = 1 score: 0.4695233133628549
WithOver: n_estim = 3, depth = 3 score: 0.5580638049242228
n_estim = 3, depth = 3 score: 0.4695233133628549
WithOver: n_estim 

KeyboardInterrupt: 

In [ ]:

from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.25, random_state=0)

spis = []
sampl = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7 , 0.8, 0.9]

depth_max = [1, 2, 3, 5, 7, 10, 25, 50, 100, 200]

n_estim = [2, 3, 5, 7, 10, 15, 20, 30, 50, 100, 150, 200, 300]

for depth1 in depth_max:
  for e1 in n_estim:
    for i1 in sampl:
      for i in sampl:

        if i1 > i:
          continue

        over = RandomOverSampler(random_state = 42, sampling_strategy=i1)

        train_x_ov, train_y_ov = over.fit_resample(X_train, y_train)

        tl = RandomUnderSampler(sampling_strategy = i)
        train_x_over, train_y_over= tl.fit_resample(train_x_ov, train_y_ov)

        model = xgb.XGBClassifier(n_estimators = e1, max_depth = depth1, tree_method = 'hist')

        model.fit(train_x_over, train_y_over)

        res = model.predict(X_test)

        print(f'depth: {depth1}, est: {e1}, over:{i1}, under:{i} - {f1_score(res, y_test, average='macro')}')

        spis.append(f1_score(res, y_test, average='macro'))


print(np.max(spis))




depth: 1, est: 2, over:0.2, under:0.2 - 0.4695233133628549
depth: 1, est: 2, over:0.2, under:0.3 - 0.4695233133628549
depth: 1, est: 2, over:0.2, under:0.4 - 0.4695233133628549
depth: 1, est: 2, over:0.2, under:0.5 - 0.4695233133628549
depth: 1, est: 2, over:0.2, under:0.6 - 0.4695233133628549
depth: 1, est: 2, over:0.2, under:0.7 - 0.5603844651025179
depth: 1, est: 2, over:0.2, under:0.8 - 0.5603844651025179
depth: 1, est: 2, over:0.2, under:0.9 - 0.5603844651025179
depth: 1, est: 2, over:0.3, under:0.3 - 0.4695233133628549
depth: 1, est: 2, over:0.3, under:0.4 - 0.4695233133628549
depth: 1, est: 2, over:0.3, under:0.5 - 0.4695233133628549
depth: 1, est: 2, over:0.3, under:0.6 - 0.4695233133628549
depth: 1, est: 2, over:0.3, under:0.7 - 0.5603844651025179
depth: 1, est: 2, over:0.3, under:0.8 - 0.5603844651025179
depth: 1, est: 2, over:0.3, under:0.9 - 0.5603844651025179
depth: 1, est: 2, over:0.4, under:0.4 - 0.4695233133628549
depth: 1, est: 2, over:0.4, under:0.5 - 0.46952331336285

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

spis = []
sampl = [0.3, 0.4, 0.5, 0.6, 0.7 , 0.8, 0.9, 1]

depth_max = [1, 2, 3, 5]

n_estim = [30, 50, 100, 150]



for depth1 in depth_max:
  for e1 in n_estim:
    for i1 in sampl:
      for i in sampl:

        if i1 > i:
          continue

        f1_full = []

        for train_idx, val_idx in cv.split(train_x, train_y):

          X_train_fold = train_x[train_idx]
          X_val_fold = train_x[val_idx]

          y_train_fold = train_y[train_idx]
          y_val_fold = train_y[val_idx]

          over = RandomOverSampler(random_state = 42, sampling_strategy=i1)

          train_x_ov, train_y_ov = over.fit_resample(X_train_fold, y_train_fold)

          tl = RandomUnderSampler(sampling_strategy = i)
          train_x_over, train_y_over= tl.fit_resample(train_x_ov, train_y_ov)

          model = xgb.XGBClassifier(n_estimators = e1, max_depth = depth1, tree_method = 'hist')

          model.fit(train_x_over, train_y_over)

          res = model.predict(X_val_fold)

          ocinka = f1_score(y_val_fold, res, average='macro')

          f1_full.append(ocinka)



        res_ocinka = np.mean(f1_full)

        print(f'depth: {depth1}, est: {e1}, over:{i1}, under:{i} - {res_ocinka}')

        spis.append(res_ocinka)


spis_copy = spis.copy()

for i in range(10):
    best_idx = np.argmax(spis_copy)
    print(f"{i+1}. f1 = {spis_copy[best_idx]:.4f}")
    spis_copy[best_idx] = -1

depth: 1, est: 2, over:0.2, under:0.2 - 0.4701515594859492
depth: 1, est: 2, over:0.2, under:0.3 - 0.4701515594859492
depth: 1, est: 2, over:0.2, under:0.4 - 0.4701515594859492
depth: 1, est: 2, over:0.2, under:0.5 - 0.4701515594859492
depth: 1, est: 2, over:0.2, under:0.6 - 0.4701515594859492
depth: 1, est: 2, over:0.2, under:0.7 - 0.5415745453338405
depth: 1, est: 2, over:0.2, under:0.8 - 0.5660850488377625
depth: 1, est: 2, over:0.2, under:0.9 - 0.5660850488377625
depth: 1, est: 2, over:0.3, under:0.3 - 0.4701515594859492
depth: 1, est: 2, over:0.3, under:0.4 - 0.4701515594859492
depth: 1, est: 2, over:0.3, under:0.5 - 0.4701515594859492
depth: 1, est: 2, over:0.3, under:0.6 - 0.4701515594859492
depth: 1, est: 2, over:0.3, under:0.7 - 0.5161227075566532
depth: 1, est: 2, over:0.3, under:0.8 - 0.5660850488377625
depth: 1, est: 2, over:0.3, under:0.9 - 0.5660850488377625
depth: 1, est: 2, over:0.4, under:0.4 - 0.4701515594859492
depth: 1, est: 2, over:0.4, under:0.5 - 0.47015155948594

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

spis = []
sampl = [0.3, 0.4, 0.5, 0.6, 0.7 , 0.8, 0.9, 1]

depth_max = [1, 2, 3, 5]

n_estim = [20, 30, 50, 100, 150]



for depth1 in depth_max:
  for e1 in n_estim:
    for i1 in sampl:
      for i in sampl:

        if i1 > i:
          continue

        f1_full = []

        for train_idx, val_idx in cv.split(train_x, train_y):

          X_train_fold = train_x[train_idx]
          X_val_fold = train_x[val_idx]

          y_train_fold = train_y[train_idx]
          y_val_fold = train_y[val_idx]

          over = RandomOverSampler(random_state = 42, sampling_strategy=i1)

          train_x_ov, train_y_ov = over.fit_resample(X_train_fold, y_train_fold)

          tl = RandomUnderSampler(sampling_strategy = i)
          train_x_over, train_y_over= tl.fit_resample(train_x_ov, train_y_ov)

          model = xgb.XGBClassifier(n_estimators = e1, max_depth = depth1, tree_method = 'hist')

          model.fit(train_x_over, train_y_over)

          res = model.predict(X_val_fold)

          ocinka = f1_score(y_val_fold, res, average='macro')

          f1_full.append(ocinka)



        res_ocinka = np.mean(f1_full)

        print(f'depth: {depth1}, est: {e1}, over:{i1}, under:{i} - {res_ocinka}')

        spis.append(res_ocinka)


spis_copy = spis.copy()

for i in range(10):
    best_idx = np.argmax(spis_copy)
    print(f"{i+1}. f1 = {spis_copy[best_idx]:.4f}")
    spis_copy[best_idx] = -1

depth: 1, est: 20, over:0.3, under:0.3 - 0.5040305527369671
depth: 1, est: 20, over:0.3, under:0.4 - 0.5313303480049106
depth: 1, est: 20, over:0.3, under:0.5 - 0.5509044918877612
depth: 1, est: 20, over:0.3, under:0.6 - 0.5667744055476945
depth: 1, est: 20, over:0.3, under:0.7 - 0.5688783891246627
depth: 1, est: 20, over:0.3, under:0.8 - 0.5616807776406358
depth: 1, est: 20, over:0.3, under:0.9 - 0.549513470065798
depth: 1, est: 20, over:0.3, under:1 - 0.5303889622671064
depth: 1, est: 20, over:0.4, under:0.4 - 0.5324315894068808
depth: 1, est: 20, over:0.4, under:0.5 - 0.5514553642154372
depth: 1, est: 20, over:0.4, under:0.6 - 0.5675585152530509
depth: 1, est: 20, over:0.4, under:0.7 - 0.5688786920110824
depth: 1, est: 20, over:0.4, under:0.8 - 0.5608722403263876
depth: 1, est: 20, over:0.4, under:0.9 - 0.5484320567008731
depth: 1, est: 20, over:0.4, under:1 - 0.5361906330565676
depth: 1, est: 20, over:0.5, under:0.5 - 0.5523237704657415
depth: 1, est: 20, over:0.5, under:0.6 - 0.56

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# train = pd.read_csv('train.csv')
# test = pd.read_csv('test.csv')


# train['f_10_iszero'] = (train['f_10'] == 0).astype(int)
# test['f_10_iszero'] = (test['f_10'] == 0).astype(int)

# train_y = train['target']
# train_x = train.drop(columns=['target', 'id'])
# test_x = test.drop(columns=['id'])


# num_cols = [col for col in train_x.columns if col.startswith('f_')]
# cat_cols = [col for col in train_x.columns if col.startswith('fc_')]

# columns_on_train = train_x.columns

# scaler = StandardScaler()
# train_num = scaler.fit_transform(train_x[num_cols])
# test_num = scaler.transform(test_x[num_cols])

# train_x = np.hstack([train_num, train_x[cat_cols].values])
# test_x = np.hstack([test_num, test_x[cat_cols].values])

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')


# 'f2_x_f4': 64, 'f_2^2': 32, 'f_4^2': 32, 'f_9^2': 32, 'f_10^2': 32, 'f_9_is_zero': 32, 'f_10_is_zero': 32


# features = [2, 4, 9, 10]

# additional_df = pd.DataFrame()


# for i in features:
#   additional_df[f'f_{i}^2'] = train[f'f_{i}'] * train[f'f_{i}']

# eps = 1e-6


# additional_df['f_2_div_f4'] = train['f_2'] / (train['f_4'] + eps)


# additional_df['f10_x_f2'] = train['f_10'] * train['f_2']

# additional_df['f10_x_f4'] = train['f_10'] * train['f_4']

# additional_df['f2_x_f4'] = train['f_2'] * train['f_4']



train['f10_x_f4'] = train['f_10'] * train['f_4']
test['f10_x_f4'] = test['f_10'] * test['f_4']

train['f4_x_f2'] = train['f_2'] * train['f_4']
test['f4_x_f2'] = test['f_2'] * test['f_4']


train['f10_x_f2'] = train['f_2'] * train['f_10']
test['f10_x_f2'] = test['f_2'] * test['f_10']

eps = 1e-6

train['f_2_div_f4'] = train['f_2'] / (train['f_4'] + eps)
test['f_2_div_f4'] = test['f_2'] / (test['f_4'] + eps)

train['f_new'] = train['f_4'] * train['f_10'] * train['f_2']
test['f_new'] = test['f_4'] * test['f_10'] * test['f_2']


num_cols = [col for col in train.columns if col.startswith('f_')]
cat_cols = [col for col in train.columns if col.startswith('fc_')]

train_y = train['target']
train_x = train.drop(columns=['target', 'id'])

columns_on_train = train_x.columns
test_x = test.drop(columns=['id'])

scaler = StandardScaler()
train_num = scaler.fit_transform(train_x[num_cols])
test_num = scaler.transform(test_x[num_cols])

train_x = np.hstack([train_num, train_x[cat_cols].values])
test_x  = np.hstack([test_num, test_x[cat_cols].values])


In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=4, shuffle=True)

f1_full = []
for train_idx, val_idx in cv.split(train_x, train_y):

  X_train_fold = train_x[train_idx]
  X_val_fold = train_x[val_idx]

  y_train_fold = train_y[train_idx]
  y_val_fold = train_y[val_idx]

  over = RandomOverSampler(random_state = 42, sampling_strategy=0.6)

  train_x_ov, train_y_ov = over.fit_resample(X_train_fold, y_train_fold)

  tl = RandomUnderSampler(sampling_strategy = 0.6, random_state=42)
  train_x_over, train_y_over= tl.fit_resample(train_x_ov, train_y_ov)

  model = xgb.XGBClassifier(n_estimators = 300, max_depth = 3, tree_method = 'hist', random_state = 42, learning_rate = 0.1)

  model.fit(train_x_over, train_y_over)

  res = model.predict(X_val_fold)

  ocinka = f1_score(y_val_fold, res, average='macro')


  f1_full.append(ocinka)

print(np.mean(f1_full))

# imp = model.feature_importances_
# for i in range(4):
#   idx = np.argmax(imp[:10])
#   print(idx)
#   imp[idx] = -10

# print(model.feature_importances_)

# print(columns_on_train[11])



0.5842960335968888


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.25, random_state=1)

over = RandomOverSampler(random_state = 42, sampling_strategy=0.6)

train_x_ov, train_y_ov = over.fit_resample(X_train, y_train)

tl = RandomUnderSampler(sampling_strategy = 0.6)
train_x_over, train_y_over= tl.fit_resample(train_x_ov, train_y_ov)

model = xgb.XGBClassifier(n_estimators = 300, max_depth = 3, tree_method = 'hist', random_state = 42, learning_rate = 0.1)

model.fit(train_x_over, train_y_over)

# res = model.predict(X_test)
importance = model.feature_importances_

imp = model.feature_importances_
for i in range(5):
  idx = np.argmax(imp[:10])
  print(f"f_{idx+1} = {imp[idx]}")
  imp[idx] = -10



# print(columns_on_train[9])
# print(columns_on_train[8])
# print(columns_on_train[3])
# print(columns_on_train[1])
print(importance)


zero_idx = np.where(importance == 0)[0]

print(zero_idx)

print(columns_on_train[zero_idx])

# print(f'{importance[9]}- {columns_on_train[9]}')


f_10 = 0.27065297961235046
f_2 = 0.07081113755702972
f_4 = 0.03396493196487427
f_9 = 0.03042672388255596
f_7 = 0.020621798932552338
[0.01777241 0.07081114 0.00954858 0.03396493 0.0189392  0.01859369
 0.0206218  0.01680858 0.03042672 0.27065298 0.02610124 0.01567884
 0.0531834  0.02229637 0.01235716 0.02351579 0.01383847 0.01920454
 0.01533313 0.02231849 0.01851236 0.01908551 0.0184091  0.01968455
 0.0260552  0.01981328 0.00883463 0.00362026 0.00735395 0.
 0.01463127 0.01522747 0.         0.01051749 0.01184868 0.0148446
 0.00460737 0.         0.         0.         0.         0.0117661
 0.         0.         0.         0.         0.         0.01271254
 0.03050821]
[29 32 37 38 39 40 42 43 44 45 46]
Index(['fc_19', 'fc_22', 'fc_27', 'fc_28', 'fc_29', 'fc_30', 'fc_32', 'fc_33',
       'fc_34', 'fc_35', 'fc_36'],
      dtype='object')


In [ ]:
over = RandomOverSampler(random_state = 42, sampling_strategy=0.6)

train_x_ov, train_y_ov = over.fit_resample(train_x, train_y)

under = RandomUnderSampler(sampling_strategy = 0.6, random_state = 42)
train_x_over, train_y_over = under.fit_resample(train_x_ov, train_y_ov)

model = xgb.XGBClassifier(n_estimators = 300, max_depth = 3, tree_method = 'hist', random_state = 42, learning_rate = 0.1)

model.fit(train_x_over, train_y_over)

res = model.predict(test_x)

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
import numpy as np

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=45)

learning_rates = [0.3, 0.2, 0.15, 0.1, 0.05]
n_estimators_list = [50, 100, 150, 200, 300]

results = []

for lr in learning_rates:
    for n_est in n_estimators_list:
        f1_full = []

        for train_idx, val_idx in cv.split(train_x, train_y):
            X_train_fold = train_x[train_idx]
            X_val_fold = train_x[val_idx]

            y_train_fold = train_y[train_idx]
            y_val_fold = train_y[val_idx]

            over = RandomOverSampler(random_state=42, sampling_strategy=0.6)
            train_x_ov, train_y_ov = over.fit_resample(X_train_fold, y_train_fold)

            under = RandomUnderSampler(random_state=42, sampling_strategy=0.6)
            train_x_bal, train_y_bal = under.fit_resample(train_x_ov, train_y_ov)

            model = xgb.XGBClassifier(
                n_estimators=n_est,
                learning_rate=lr,
                max_depth=3,
                tree_method='hist',
                random_state=42,
                n_jobs=1
            )

            model.fit(train_x_bal, train_y_bal)

            pred = model.predict(X_val_fold)
            score = f1_score(y_val_fold, pred, average='macro')
            f1_full.append(score)

        mean_score = np.mean(f1_full)
        results.append((lr, n_est, mean_score))
        print(f"lr={lr}, n_estimators={n_est}, mean_f1={mean_score:.5f}")

best = max(results, key=lambda x: x[2])
print("\nBest:")
print(f"learning_rate={best[0]}, n_estimators={best[1]}, mean_f1={best[2]:.5f}")

lr=0.3, n_estimators=50, mean_f1=0.58190
lr=0.3, n_estimators=100, mean_f1=0.58357
lr=0.3, n_estimators=150, mean_f1=0.58200
lr=0.3, n_estimators=200, mean_f1=0.58050
lr=0.3, n_estimators=300, mean_f1=0.57480
lr=0.2, n_estimators=50, mean_f1=0.58080
lr=0.2, n_estimators=100, mean_f1=0.58189
lr=0.2, n_estimators=150, mean_f1=0.58264
lr=0.2, n_estimators=200, mean_f1=0.58184
lr=0.2, n_estimators=300, mean_f1=0.58030
lr=0.15, n_estimators=50, mean_f1=0.57989
lr=0.15, n_estimators=100, mean_f1=0.58283
lr=0.15, n_estimators=150, mean_f1=0.58396
lr=0.15, n_estimators=200, mean_f1=0.58261
lr=0.15, n_estimators=300, mean_f1=0.58340
lr=0.1, n_estimators=50, mean_f1=0.57960
lr=0.1, n_estimators=100, mean_f1=0.58167
lr=0.1, n_estimators=150, mean_f1=0.58392
lr=0.1, n_estimators=200, mean_f1=0.58473
lr=0.1, n_estimators=300, mean_f1=0.58442
lr=0.05, n_estimators=50, mean_f1=0.57267
lr=0.05, n_estimators=100, mean_f1=0.57835
lr=0.05, n_estimators=150, mean_f1=0.58055
lr=0.05, n_estimators=200, mean

In [ ]:
features = [2, 4, 9, 10]

additional_df = pd.DataFrame()


for i in features:
  additional_df[f'f_{i}^2'] = train[f'f_{i}'] * train[f'f_{i}']

eps = 1e-6

additional_df['f_10_div_f2'] = train['f_10'] / (train['f_2'] + eps)

additional_df['f_10_div_f4'] = train['f_10'] / (train['f_4'] + eps)


additional_df['f_2_div_f4'] = train['f_2'] / (train['f_4'] + eps)


additional_df['f10_x_f2'] = train['f_10'] * train['f_2']


additional_df['f10_x_f4'] = train['f_10'] * train['f_4']


additional_df['f2_x_f4'] = train['f_2'] * train['f_4']


additional_df['f_9_is_zero'] = (train['f_9'] == 0).astype(int)


additional_df['f_10_is_zero'] = (train['f_10'] == 0).astype(int)

print(additional_df)



       f_2^2  f_4^2  f_9^2  f_10^2  f_10_div_f2  f_10_div_f4  f_2_div_f4  \
0        625      1      0       0     0.000000     0.000000   24.999975   
1          1      9      0       0     0.000000     0.000000    0.333333   
2          1      4      0       1     0.999999     0.500000    0.500000   
3          1      4      0       0     0.000000     0.000000    0.500000   
4          1      1      0       0     0.000000     0.000000    0.999999   
...      ...    ...    ...     ...          ...          ...         ...   
65160      1      4      1       1     0.999999     0.500000    0.500000   
65161      1     25      0       1     0.999999     0.200000    0.200000   
65162      1      1      0       0     0.000000     0.000000    0.999999   
65163      1     36      1       4     1.999998     0.333333    0.166667   
65164      1     36      0       0     0.000000     0.000000    0.166667   

       f10_x_f2  f10_x_f4  f2_x_f4  f_9_is_zero  f_10_is_zero  
0             0        

In [ ]:
from itertools import combinations

new_features = [
    'f_2^2',
    'f_4^2',
    'f_9^2',
    'f_10^2',
    'f_10_div_f2',
    'f_10_div_f4',
    'f_2_div_f4',
    'f10_x_f2',
    'f10_x_f4',
    'f2_x_f4',
    'f_9_is_zero',
    'f_10_is_zero'
]

all_combinations = []

for r in range(1, len(new_features) + 1):   # r = 1, 2, 3, ..., 12
    for comb in combinations(new_features, r):
        all_combinations.append(list(comb))




Кількість усіх комбінацій: 4095
[['f_2^2'], ['f_4^2'], ['f_9^2'], ['f_10^2'], ['f_10_div_f2'], ['f_10_div_f4'], ['f_2_div_f4'], ['f10_x_f2'], ['f10_x_f4'], ['f2_x_f4'], ['f_9_is_zero'], ['f_10_is_zero'], ['f_2^2', 'f_4^2'], ['f_2^2', 'f_9^2'], ['f_2^2', 'f_10^2']]


In [ ]:

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

result = []

for i in all_combinations:

  df = np.hstack([train_x, additional_df[i]])

  f1_full = []
  for train_idx, val_idx in cv.split(df, train_y):

    X_train_fold = df[train_idx]
    X_val_fold = df[val_idx]

    y_train_fold = train_y[train_idx]
    y_val_fold = train_y[val_idx]

    over = RandomOverSampler(random_state = 42, sampling_strategy=0.6)

    train_x_ov, train_y_ov = over.fit_resample(X_train_fold, y_train_fold)

    tl = RandomUnderSampler(sampling_strategy = 0.6, random_state=42)
    train_x_over, train_y_over= tl.fit_resample(train_x_ov, train_y_ov)

    model = xgb.XGBClassifier(n_estimators = 100, max_depth = 3, tree_method = 'hist', random_state = 42, n_jobs=1)

    model.fit(train_x_over, train_y_over)

    res = model.predict(X_val_fold)

    ocinka = f1_score(y_val_fold, res, average='macro')

    f1_full.append(ocinka)


  f1 = np.mean(f1_full)

  print(f1, i)

  result.append(f1)



print(np.max(result))





0.5831886734924164 ['f_2^2']
0.5831886734924164 ['f_4^2']
0.5831886734924164 ['f_9^2']
0.5831886734924164 ['f_10^2']
0.5840203931449215 ['f_10_div_f2']
0.5830757825852058 ['f_10_div_f4']
0.5842919407650171 ['f_2_div_f4']
0.5824467023650114 ['f10_x_f2']
0.5838451299081409 ['f10_x_f4']
0.5850709376067953 ['f2_x_f4']
0.5831886734924164 ['f_9_is_zero']
0.5831886734924164 ['f_10_is_zero']
0.5831886734924164 ['f_2^2', 'f_4^2']
0.5831886734924164 ['f_2^2', 'f_9^2']
0.5831886734924164 ['f_2^2', 'f_10^2']
0.5840203931449215 ['f_2^2', 'f_10_div_f2']
0.5830757825852058 ['f_2^2', 'f_10_div_f4']
0.5842919407650171 ['f_2^2', 'f_2_div_f4']
0.5824467023650114 ['f_2^2', 'f10_x_f2']
0.5838451299081409 ['f_2^2', 'f10_x_f4']
0.5850709376067953 ['f_2^2', 'f2_x_f4']
0.5831886734924164 ['f_2^2', 'f_9_is_zero']
0.5831886734924164 ['f_2^2', 'f_10_is_zero']
0.5831886734924164 ['f_4^2', 'f_9^2']
0.5831886734924164 ['f_4^2', 'f_10^2']
0.5840203931449215 ['f_4^2', 'f_10_div_f2']
0.5830757825852058 ['f_4^2', 'f_10_

In [ ]:
indexs = [i for i, x in enumerate(result) if abs(x - 0.5850709376067953) < 1e-9]

In [ ]:
optimal_features = [all_combinations[i] for i in indexs]

In [ ]:
from collections import Counter


# Зливаємо всі підсписки в один список
all_features = [feat for combo in optimal_features for feat in combo]

# Рахуємо кількість появ кожної фічі
feature_counts = Counter(all_features)

print(feature_counts)

Counter({'f2_x_f4': 64, 'f_2^2': 32, 'f_4^2': 32, 'f_9^2': 32, 'f_10^2': 32, 'f_9_is_zero': 32, 'f_10_is_zero': 32})


In [ ]:
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=41)

results = []

for comb in optimal_features:
    df = train_x

    f1_full = []

    for train_idx, val_idx in cv.split(df, train_y):
        X_train_fold = df[train_idx]
        X_val_fold = df[val_idx]

        y_train_fold = train_y[train_idx]
        y_val_fold = train_y[val_idx]

        over = RandomOverSampler(random_state=42, sampling_strategy=0.6)
        train_x_ov, train_y_ov = over.fit_resample(X_train_fold, y_train_fold)

        tl = RandomUnderSampler(sampling_strategy=0.6, random_state=42)
        train_x_over, train_y_over = tl.fit_resample(train_x_ov, train_y_ov)

        model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=3,
            tree_method='hist',
            random_state=42
        )

        model.fit(train_x_over, train_y_over)
        res = model.predict(X_val_fold)

        score = f1_score(y_val_fold, res, average='macro')
        f1_full.append(score)

    mean_f1 = np.mean(f1_full)
    std_f1 = np.std(f1_full)

    results.append({
        'combination': comb,
        'mean_f1': mean_f1,
        'std_f1': std_f1,
        'num_features': len(comb),
        'fold_scores': f1_full
    })

# сортуємо:
# 1) по mean_f1 за спаданням
# 2) по std_f1 за зростанням
# 3) по num_features за зростанням
results = sorted(results, key=lambda x: (-x['mean_f1'], x['std_f1'], x['num_features']))

best = results[0]

print("Best combination:", best['combination'])
print("Mean F1:", best['mean_f1'])
print("Std F1:", best['std_f1'])
print("Fold scores:", best['fold_scores'])

Best combination: ['f2_x_f4']
Mean F1: 0.5812676129334138
Std F1: 0.002920617831251312
Fold scores: [0.5839470828425597, 0.5820373909429113, 0.5827366750329304, 0.5763493029152539]


In [ ]:
features = [2, 4, 9, 10]

additional_df = pd.DataFrame()


for i in features:
  additional_df[f'f_{i}^2'] = train[f'f_{i}'] * train[f'f_{i}']

eps = 1e-6


additional_df['f_2_div_f4'] = train['f_2'] / (train['f_4'] + eps)


additional_df['f10_x_f2'] = train['f_10'] * train['f_2']

additional_df['f10_x_f4'] = train['f_10'] * train['f_4']

additional_df['f2_x_f4'] = train['f_2'] * train['f_4']


In [ ]:
from itertools import combinations
import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

train = pd.read_csv('train.csv')
train_x = train.drop(columns=['target', 'id']).values
train_y = train['target'].values


cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

learning_rates = [0.3, 0.2, 0.15, 0.1, 0.05]
n_estimators_list = [50, 100, 150, 200, 300]
over_sampl = [0.2 , 0.6, 0.9, 1]
under_sampl = [0.2, 0.6, 0.9, 1]

depth_max = [1, 2, 3, 5]

new_features = [
    'f_2_div_f4',
    'f10_x_f2',
    'f10_x_f4',
    'f2_x_f4',
]

all_combinations = []
for r in range(1, len(new_features) + 1):
    for comb in combinations(new_features, r):
        all_combinations.append(list(comb))

results = []

for feat_comb in all_combinations:

    df = np.hstack([train_x, additional_df[feat_comb].values])

    for lr in learning_rates:
        for n_est in n_estimators_list:
            for over_s in over_sampl:
                for under_s in under_sampl:




                  if over_s > under_s:
                    continue

                  for depth in depth_max:

                        f1_full = []

                        for train_idx, val_idx in cv.split(df, train_y):

                            X_train_fold = df[train_idx]
                            X_val_fold = df[val_idx]

                            y_train_fold = train_y[train_idx]
                            y_val_fold = train_y[val_idx]

                            over = RandomOverSampler(random_state=42, sampling_strategy=over_s)
                            X_over, y_over = over.fit_resample(X_train_fold, y_train_fold)

                            under = RandomUnderSampler(random_state=42, sampling_strategy=under_s)
                            X_bal, y_bal = under.fit_resample(X_over, y_over)

                            model = xgb.XGBClassifier(
                                n_estimators=n_est,
                                learning_rate=lr,
                                max_depth=depth,
                                tree_method='hist',
                                random_state=42,
                                n_jobs=1
                            )

                            model.fit(X_bal, y_bal)

                            pred = model.predict(X_val_fold)
                            score = f1_score(y_val_fold, pred, average='macro')

                            f1_full.append(score)

                        mean_f1 = np.mean(f1_full)

                        print(f"F1={mean_f1:.5f} | feats={feat_comb} | lr={lr} | n_est={n_est} | over={over_s} | under={under_s} | depth={depth}")

                        results.append({
                            'f1': mean_f1,
                            'features': feat_comb,
                            'lr': lr,
                            'n_estimators': n_est,
                            'over_sampling': over_s,
                            'under_sampling': under_s,
                            'max_depth': depth
                        })

results_sorted = sorted(results, key=lambda x: x['f1'], reverse=True)

print("\nTOP 10")
for i in range(10):
    r = results_sorted[i]
    print(f"{i+1}) F1={r['f1']:.5f} | feats={r['features']} | lr={r['lr']} | n_est={r['n_estimators']} | over={r['over_sampling']} | under={r['under_sampling']} | depth={r['max_depth']}")

Выходные данные были обрезаны до нескольких последних строк (5000).
F1=0.54143 | feats=['f_2_div_f4', 'f2_x_f4'] | lr=0.1 | n_est=150 | over=0.9 | under=1 | depth=5
F1=0.53575 | feats=['f_2_div_f4', 'f2_x_f4'] | lr=0.1 | n_est=150 | over=1 | under=1 | depth=1
F1=0.52211 | feats=['f_2_div_f4', 'f2_x_f4'] | lr=0.1 | n_est=150 | over=1 | under=1 | depth=2
F1=0.52557 | feats=['f_2_div_f4', 'f2_x_f4'] | lr=0.1 | n_est=150 | over=1 | under=1 | depth=3
F1=0.53796 | feats=['f_2_div_f4', 'f2_x_f4'] | lr=0.1 | n_est=150 | over=1 | under=1 | depth=5
F1=0.48292 | feats=['f_2_div_f4', 'f2_x_f4'] | lr=0.1 | n_est=200 | over=0.2 | under=0.2 | depth=1
F1=0.49345 | feats=['f_2_div_f4', 'f2_x_f4'] | lr=0.1 | n_est=200 | over=0.2 | under=0.2 | depth=2
F1=0.49799 | feats=['f_2_div_f4', 'f2_x_f4'] | lr=0.1 | n_est=200 | over=0.2 | under=0.2 | depth=3
F1=0.50503 | feats=['f_2_div_f4', 'f2_x_f4'] | lr=0.1 | n_est=200 | over=0.2 | under=0.2 | depth=5
F1=0.57281 | feats=['f_2_div_f4', 'f2_x_f4'] | lr=0.1 | n_e

In [ ]:
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

f1_full = []
for train_idx, val_idx in cv.split(train_x, train_y):

  X_train_fold = train_x[train_idx]
  X_val_fold = train_x[val_idx]

  y_train_fold = train_y[train_idx]
  y_val_fold = train_y[val_idx]

  over = RandomOverSampler(random_state = 42, sampling_strategy=0.6)

  train_x_ov, train_y_ov = over.fit_resample(X_train_fold, y_train_fold)

  tl = RandomUnderSampler(sampling_strategy = 0.6, random_state=42)
  train_x_over, train_y_over= tl.fit_resample(train_x_ov, train_y_ov)

  model = xgb.XGBClassifier(n_estimators = 100, max_depth = 3, tree_method = 'hist', random_state = 42, n_jobs=1)

  model.fit(train_x_over, train_y_over)

  res = model.predict(X_val_fold)

  ocinka = f1_score(y_val_fold, res, average='macro')

  f1_full.append(ocinka)

print(np.mean(f1_full))

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier


X, X_test, y, y_test = train_test_split(train_x, train_y, test_size=0.20, random_state=1)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=1)


over = RandomOverSampler(random_state = 42, sampling_strategy=0.3)

train_x_ov, train_y_ov = over.fit_resample(X_train, y_train)

under = RandomUnderSampler(sampling_strategy = 0.6)
train_x_over, train_y_over = under.fit_resample(train_x_ov, train_y_ov)


gbm = AdaBoostClassifier(n_estimators=100, estimator=DecisionTreeClassifier(max_depth=3), random_state=1)

model = xgb.XGBClassifier(n_estimators = 100, max_depth = 3, tree_method = 'hist')

gbm.fit(train_x_over, train_y_over)
model.fit(train_x_over, train_y_over)

gbm_res = gbm.predict(X_val)
model_res = model.predict(X_val)

ocinka_gbm = f1_score(y_val, gbm_res, average='macro')
ocinka_xgb = f1_score(y_val, model_res, average='macro')

print(ocinka_gbm)
print(ocinka_xgb)

gbm_res_proba = gbm.predict_proba(X_test)
xgb_res_proba = model.predict_proba(X_test)

ensemble_proba = (ocinka_gbm / (ocinka_gbm + ocinka_xgb)) * gbm_res_proba + (ocinka_xgb / (ocinka_gbm + ocinka_xgb)) * xgb_res_proba

res = np.argmax(ensemble_proba, axis = 1)

ocinka = f1_score(y_test, res, average='macro')

print(ocinka)

KeyboardInterrupt: 

In [ ]:
import pandas as pd

# чтобы pandas не обрезал вывод
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

# допустим, results — это список словарей, который ты наполняешь в цикле
# пример:
results = []

# превращаем в DataFrame
df_results = pd.DataFrame(results)

# сортируем по F1 от лучшего к худшему
df_results = df_results.sort_values('f1', ascending=False).reset_index(drop=True)

# вывести все строки
print(df_results.to_string(index=False))

NameError: name 'mean_f1' is not defined

In [ ]:
from sklearn.ensemble import StackingClassifier

X, X_test, y, y_test = train_test_split(train_x, train_y, test_size=0.25, random_state=1)

over = RandomOverSampler(random_state = 42, sampling_strategy=0.6)

train_x_ov, train_y_ov = over.fit_resample(train_x, train_y)

under = RandomUnderSampler(sampling_strategy = 0.6)
train_x_over, train_y_over = under.fit_resample(train_x_ov, train_y_ov)


estimators = [
    ('decision_tree', xgb.XGBClassifier(n_estimators = 100, max_depth = 3, tree_method = 'hist')),
    ('log_reg', AdaBoostClassifier(n_estimators = 100, estimator=DecisionTreeClassifier(max_depth=3), random_state=1))
]



model = StackingClassifier(estimators)

model.fit(train_x, train_y)

res = model.predict(test_x)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# X: [N, D]
# truth: [N, K]

X_t = torch.tensor(train_x, dtype=torch.float32)
y_t = torch.tensor(train_y, dtype=torch.long)

D = train_x.shape[1]
K = len(np.unique(train_y))
h = 100

model = nn.Sequential(
    nn.Linear(D, h),
    nn.Tanh(),
    nn.Linear(h, h),
    nn.Tanh(),
    nn.Linear(h, K)
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=1e-3)

for i in range(2000):
    optimizer.zero_grad()         # обнуляємо старі градієнти
    outputs = model(X_t)          # forward
    loss = criterion(outputs, y_t)
    loss.backward()               # backprop автоматично
    optimizer.step()              # оновлення ваг

    if i % 100 == 0:
        print(f"iteration {i}: loss {loss.item():.6e}")

KeyboardInterrupt: 

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

f1_full = []
for train_idx, val_idx in cv.split(train_x, train_y):

  X_train_fold = train_x[train_idx]
  X_val_fold = train_x[val_idx]

  y_train_fold = train_y[train_idx]
  y_val_fold = train_y[val_idx]

  over = RandomOverSampler(random_state = 42, sampling_strategy=0.6)

  train_x_ov, train_y_ov = over.fit_resample(X_train_fold, y_train_fold)

  tl = RandomUnderSampler(sampling_strategy = 0.6)
  train_x_over, train_y_over= tl.fit_resample(train_x_ov, train_y_ov)

  X_t = torch.tensor(np.asarray(train_x_over), dtype=torch.float32)
  y_t = torch.tensor(np.asarray(train_y_over), dtype=torch.long)

  x_val = torch.tensor(np.asarray(X_val_fold), dtype=torch.float32)

  D = X_train_fold.shape[1]
  K = len(np.unique(y_train_fold))
  h = 100

  model = nn.Sequential(
      nn.Linear(D, h),
      nn.Tanh(),
      nn.Linear(h, h),
      nn.Tanh(),
      nn.Softmax(h, K)
  )

  criterion = nn.CrossEntropyLoss()
  optimizer = optim.SGD(model.parameters(), lr=1e-3)

  for i in range(100):
      optimizer.zero_grad()         # обнуляємо старі градієнти
      outputs = model(X_t)          # forward
      loss = criterion(outputs, y_t)
      loss.backward()               # backprop автоматично
      optimizer.step()              # оновлення ваг

      if i % 100 == 0:
          print(f"iteration {i}: loss {loss.item():.6e}")

  with torch.no_grad():
    res = model(x_val)
    preds = torch.argmax(res, dim=1)

  ocinka = f1_score(y_val_fold, preds.numpy(), average='macro')

  f1_full.append(ocinka)

print(np.mean(f1_full))

iteration 0: loss 6.969156e-01
iteration 0: loss 7.769583e-01
iteration 0: loss 7.526635e-01
iteration 0: loss 6.924965e-01
iteration 0: loss 6.961074e-01
iteration 0: loss 6.892371e-01
0.48222798488876467


In [ ]:
submission = test[['id']].copy()

submission['target'] = res.astype(int)

output_path = 'submission_from_notebook.csv'

submission.to_csv(output_path, index=False)

In [ ]:
files.upload();

Saving kaggle.json to kaggle.json


In [ ]:
import json

!mkdir /root/.kaggle/
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!kaggle config set -n path -v{/content}

mkdir: cannot create directory ‘/root/.kaggle/’: File exists
- path is now set to: {/content}


In [ ]:
!kaggle competitions submit -c ml-kse-2026 -f submission_from_notebook.csv -m "perebitie bonda"

100% 222k/222k [00:00<00:00, 300kB/s]
Successfully submitted to ML@KSE2026